# പാഠം 04 - ടൂൾ ഉപയോഗ മാതൃക

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework (Python) ഉപയോഗിച്ച് AI ഏജന്റുകൾ‍ക്കുള്ള **ടൂൾ ഉപയോഗ** ഡിസൈൻ മാതൃക പഠിക്കും. നമ്മൾ ഉൾക്കൊള്ളുന്നതാണ്:

- `@tool` ഡെക്കറേറ്ററും ടൈപ്പുചെയ്ത പാരാമീറ്ററുകളും ഉപയോഗിച്ച് ഫംഗ്ഷൻ ടൂളുകൾ നിർവചിക്കൽ
- ഓരോ ടൂളും എന്ത് ചെയ്യുന്നതാൻമോ മോഡലിന് അറിയിക്കാൻ ടൂൾ സ്‌കീമകൾ നൽകൽ
- `approval_mode` ഉപയോഗിച്ച് ടൂൾ പ്രവർത്തനം നിയന്ത്രിക്കൽ
- Pydantic മോഡലുകളും `response_format` ഉപയോഗിച്ച് **ഘടനാപരമായ ഫലങ്ങൾ** മടക്കുക

സാന്ദർഭ്യം ഒരു **പ്രവാസ ബുക്ക് ചെയ്യൽ ഏജന്റ്** ആണ്, അത് ഗമ്യസ്ഥാനങ്ങൾ കണ്ടെത്താനും, ലഭ്യത പരിശോധിക്കാനും, ഫ്ലൈറ്റ് വിവരങ്ങൾ തിരയാനുമുള്ളത്.


## ക്രമീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ഡെക്കറേറ്ററോടുകൂടി ടൂളുകൾ നിർവ്വചിക്കൽ

`@tool` ഡെക്കറേറ്റർ ഒരു സാധാരണ പൈത്തൺ ഫംഗ്ഷനെ ഏജന്റ് വിളിക്കunna ടൂളായി മാറ്റുന്നു.
പ്രധാന കാര്യങ്ങൾ:

- **docstring** മോഡൽ കാണുന്ന ടൂൾ വിവരണമാകും.
- **ടൈപ്പ് അനോട്ടേഷനുകൾ** (`Annotated` വിവരണങ്ങളടങ്ങിയവ ഉൾപ്പെടെ) ടൂൾ സ്കീമ നിർവ്വചിക്കുന്നു.
- `approval_mode` ഓരോ കോളിനുമുമുമ്പ് ഉപയോക്താവ് അംഗീകാരം നൽകേണ്ടതുണ്ടോ എന്ന് നിയന്ത്രിക്കുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## പലയിടങ്ങളിൽ ഉപകരണങ്ങളോടുകൂടിയ ഒരു ഏജന്റ് സൃഷ്ടിക്കൽ

ഉപയോക്തൃ ചോദ്യം മറുപടി നൽകാൻ മോഡൽ ആവശ്യപ്പെടുന്ന ഏതു ഉപകരണവും വിളിക്കാനായി മൂന്നു ഉപകരണങ്ങളും ക്ലയന്റിന് നൽകുക.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ടൂളുകളോടു കൂടിയ ഘടനാബദ്ധമായ ഔട്ട്‌പുട്ട്

`response_format` ഒരു പൈഡാന്റ്റിക് മോഡലിലേക്ക് സജ്ജമാക്കിയാൽ, ഏജന്റ് ഫ്രീ-ഫോം ടെക്സ്റ്റ് പകരം നന്നായി ടൈപ്പ് ചെയ്ത JSON ഒബ്‌ജക്റ്റ് മാറിവിളയ്ക്കാൻ നിർബന്ധിതനാണ്. താഴെത്തുടരുന്ന കോഡ് ഫലത്തെ പ്രോഗ്രാമാറ്റിക്കായി ഉപയോഗിക്കേണ്ടപ്പോൾ ഇത് ഉപകാരപ്രദമാണ്.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ഉപകരണ അംഗീകൃത മാതൃകകൾ

`@tool`-ൽ ഉള്ള `approval_mode` പാരാമീറ്റർ, ഉപകരണം പ്രവർത്തിക്കാൻ മുമ്പ് മനുഷ്യ അംഗീകാരം ആവശ്യമാണോ എന്നത് നിയന്ത്രിക്കുന്നു:

| മോട് | പെരുമാറ്റം |
|---|---|
| `"never_require"` | ഉപകരണം സ്വയമേവ പ്രവർത്തിക്കും — ഉപയോക്തൃ സ്ഥിരീകരണം ആവശ്യമില്ല. |
| `"always_require"` | ഓരോ കോഴും പ്രവർത്തിപ്പിക്കുന്നതിന് മുമ്പ് ഉപയോക്താവിൻ്റെ അംഗീകാരം വേണം. |

ഫ്ലൈറ്റ് ബുക്കിംഗ്, ക്രെഡിറ്റ് കാർഡ് ചാർജിങ് പോലുള്ള സൈഡ്-ഇഫക്ടുകൾ ഉള്ള ഉപകരണങ്ങൾക്ക് `"always_require"` ഉപയോഗിക്കുക, ഇതിലൂടെ മനുഷ്യൻ പ്രക്രിയയിൽ തുടരാൻ കഴിയും.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചതു:

1. **ഉപകരണം നിർവ്വചിക്കുക** ടൈപ്പഡ് പാരാമീറ്ററുകളും ഉപകരണം സ്കീമയായി സേവനമProvideച docstrings ഉം ഉള്ള `@tool` ഡെക്കറേറ്റർ ഉപയോഗിച്ച്.
2. **മൾട്ടിപ്പിൾ ടൂളുകൾ സമന്വയിപ്പിക്കുക** ताकि ഏജൻറ് സങ്കീർണ്ണ ചോദ്യങ്ങൾക്ക് ക(seq)വാജബ് നൽകാൻ അവയെ ക്രമത്തിൽ വിളിക്കാനാകും.
3. **സംഘടിത ഔട്ട്‌പുട്ട് തിരിച്ചു നൽകുക** `response_format` ആയി Pydantic മോഡൽ കൈമാറി.
4. **ഉപകരണ അംഗീകാരം നിയന്ത്രിക്കുക** `approval_mode` ഉപയോഗിച്ച് സുസ്ഥിരമായ പ്രവർത്തനങ്ങൾക്ക് മനുഷ്യരായ നിയന്ത്രണം കൈവരിക്കാൻ.

ഈ മാതൃകകൾ വിശ്വസ്തമായ, ഉത്പാദനോൽപ്പന്നം തയ്യാറാക്കിയ ഏജന്റുകൾക്ക് അടിസ്ഥാനമാകുന്നു, അവ സുരക്ഷിതമായി ബാഹ്യ സിസ്റ്റങ്ങൾ ബന്ധപ്പെടാൻ കഴിയും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
